# NeuroContour — 3D Extension (v2)

Reassembles the winning model's per-slice test predictions into native 240x240x155
volumes for inspection in MRIcroGL. These are a **2D model's predictions reassembled** —
not a volumetric model. Slices the project never evaluated stay empty (0) in the
prediction volume: they are **NOT EVALUATED**, never model-negative. Test patients only.

This version runs the main notebook's own code — preprocessing, dataset, model, and
Dice are copied verbatim from `unet_brats_segmentation.ipynb` — so there is no
re-implementation anywhere in the path.

In [1]:
from pathlib import Path
import random
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DATA_ROOT = Path("data")
MIN_TUMOR_PIXELS = 50
BATCH_SIZE = 8

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

OUT_3D = Path("outputs/3d")
OUT_3D.mkdir(parents=True, exist_ok=True)

case_dirs = sorted([e for e in DATA_ROOT.iterdir() if e.name.startswith("BraTS20_Training_")],
                   key=lambda e: e.name)
print("cases found:", len(case_dirs))

TEST_CASES = ["BraTS20_Training_068", "BraTS20_Training_074", "BraTS20_Training_005",
              "BraTS20_Training_055", "BraTS20_Training_012", "BraTS20_Training_014",
              "BraTS20_Training_018", "BraTS20_Training_029", "BraTS20_Training_032",
              "BraTS20_Training_036", "BraTS20_Training_004", "BraTS20_Training_015"]

Device: mps
cases found: 75


**Verbatim from the main notebook** — code cell 12 (loading helpers):

In [2]:
def prep_scan(slice_2d):
    # in: one raw scan slice (240x240 numbers)
    t = torch.from_numpy(slice_2d.astype(np.float32))
    t = t.unsqueeze(0).unsqueeze(0)
    small = F.interpolate(t, size=(128, 128), mode="bilinear", align_corners=False)
    small = small.squeeze(0).squeeze(0)

    # store tensor's mean and std
    tensor_mean = small.mean()
    tensor_std = float(small.std())
    if tensor_std == 0:
        tensor_std = 1.0
    normed = (small - tensor_mean) / tensor_std
    return normed


def prep_mask(slice_2d):
    m = torch.from_numpy(slice_2d.astype(np.float32))
    m = m.unsqueeze(0).unsqueeze(0)
    reduced = F.interpolate(m, size=(128, 128), mode="nearest")
    reduced = reduced.squeeze(0).squeeze(0)
    return reduced

**Verbatim** — code cell 13 (the slice dataset):

In [3]:
class SliceDataset(Dataset):
    def __init__(self, examples, flip=False):
        self.examples = examples
        self.flip = flip
        self.folders = {}
        for p in case_dirs:
            self.folders[p.name] = p

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        name, z = self.examples[i]
        folder = self.folders[name]
        tensors = []

        # loading in volumes and slicing at z
        for key in ["t1", "t1ce", "t2", "flair"]:
            volume = nib.load(folder / f"{name}_{key}.nii").get_fdata()
            tensors.append(prep_scan(volume[:, :, z]))

        seg = nib.load(folder / f"{name}_seg.nii").get_fdata()
        seg_n = seg[:, :, z]

        x = torch.stack(tensors, dim=0)
        y = prep_mask(seg_n > 0).unsqueeze(0)

        if self.flip and random.random() < 0.5:
            x = torch.flip(x, dims=[2])
            y = torch.flip(y, dims=[2])

        return x, y

**Verbatim** — code cells 15 and 16 (the building block and the U-Net):

In [4]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.network(x)

In [5]:
class UNet(nn.Module):
    def __init__(self, use_skips=True):
        super().__init__()
        self.use_skips = use_skips

        # Encoder block
        self.enc1 = DoubleConv(4, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(256, 512)

        # Down Sizing
        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.up4 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.head = nn.Conv2d(32, 1, kernel_size=1)

        # Decoder block
        self.dec1 = DoubleConv(512 if use_skips else 256, 256)
        self.dec2 = DoubleConv(256 if use_skips else 128, 128)
        self.dec3 = DoubleConv(128 if use_skips else 64, 64)
        self.dec4 = DoubleConv(64 if use_skips else 32, 32)

    def forward(self, x):
        # down, saving skips
        s1 = self.enc1(x)
        x = self.pool(s1)
        s2 = self.enc2(x)
        x = self.pool(s2)
        s3 = self.enc3(x)
        x = self.pool(s3)
        s4 = self.enc4(x)
        x = self.pool(s4)
        x = self.bottleneck(x)

        # up, then out
        x = self.up1(x)
        if self.use_skips:
            x = torch.cat([x, s4], dim=1)
        x = self.dec1(x)

        x = self.up2(x)
        if self.use_skips:
            x = torch.cat([x, s3], dim=1)
        x = self.dec2(x)

        x = self.up3(x)
        if self.use_skips:
            x = torch.cat([x, s2], dim=1)
        x = self.dec3(x)

        x = self.up4(x)
        if self.use_skips:
            x = torch.cat([x, s1], dim=1)
        x = self.dec4(x)

        return self.head(x)

**Verbatim** — the Dice score from code cell 18:

In [6]:
def dice_score(logits, target):
    t = target.reshape(target.size(0), -1)
    pred = (torch.sigmoid(logits) >= 0.5).float()
    pred = pred.reshape(pred.size(0), -1)
    inter = (pred * t).sum(dim=1)
    denom = pred.sum(dim=1) + t.sum(dim=1)
    scores = []
    for i in range(pred.size(0)):
        if denom[i] == 0:
            scores.append(1.0)
        else:
            scores.append((2 * inter[i] / denom[i]).item())
    return sum(scores) / len(scores)

Load the winner — the same resurrection as code cell 28, with the winner fixed (dice_loss, skips on):

In [7]:
model = UNet(use_skips=True).to(device)
model.load_state_dict(torch.load("dice_loss_best.pt", map_location=device))
model.eval()
print("winner loaded, parameters:", sum(p.numel() for p in model.parameters()))

winner loaded, parameters: 7766273


## Predict one patient into a volume

The score printed here is computed **exactly as code cell 28 computed it** — the same
dataset, the same `dice_score`, averaged per slice at 128x128 — so it must reproduce
cell 28's per-patient numbers. That is the gate: if it matches, the predictions being
placed into the volume are provably the winner's own.

(Scoring the finished volume at 240x240 instead reads lower — the native-resolution
truth carries boundary detail a 128x128 prediction cannot match pixel-for-pixel.
That is a resolution effect, not a different model.)

In [8]:
def load_seg(case):
    img = nib.load(DATA_ROOT / case / f"{case}_seg.nii")
    return img.get_fdata(), img

def evaluated_slices(seg):
    # the same rule that built the index: tumor pixels >= 50 on the slice
    return [z for z in range(seg.shape[2]) if (seg[:, :, z] > 0).sum() >= MIN_TUMOR_PIXELS]

def predict_case(case):
    seg, seg_img = load_seg(case)
    truth = (seg > 0).astype(np.uint8)
    zs = evaluated_slices(seg)
    entries = [(case, z) for z in zs]
    ds = SliceDataset(entries, flip=False)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, multiprocessing_context="fork")

    pred = np.zeros((240, 240, 155), dtype=np.uint8)   # non-evaluated slices stay 0 = NOT EVALUATED
    error = np.zeros((240, 240, 155), dtype=np.uint8)  # 0 not evaluated, 1 agree, 2 missed, 3 false positive
    running_dice = 0.0
    slice_count = 0
    i = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            running_dice += dice_score(logits, y) * x.size(0)
            slice_count += x.size(0)
            hard = (torch.sigmoid(logits) >= 0.5).float().cpu()
            for b in range(hard.size(0)):
                z = entries[i][1]
                i += 1
                up = F.interpolate(hard[b:b + 1], size=(240, 240), mode="nearest")
                p = up[0, 0].numpy().astype(np.uint8)
                pred[:, :, z] = p
                t = truth[:, :, z]
                e = np.ones((240, 240), dtype=np.uint8)
                e[(t == 1) & (p == 0)] = 2
                e[(t == 0) & (p == 1)] = 3
                error[:, :, z] = e

    score = running_dice / slice_count
    print(case, "slices:", slice_count, "mean Dice (cell-28 statistic):", round(score, 4))
    return pred, error, seg_img, score

SCORES = {}
pred, error, seg_img, score = predict_case("BraTS20_Training_055")
SCORES["BraTS20_Training_055"] = score

BraTS20_Training_055 slices: 58 mean Dice (cell-28 statistic): 0.8848


**Gate:** the print above must read **0.8848** (cell 28: 0.884770545980026), give or take a thousandth of float dust. If it does, save 055's volumes:

In [9]:
nib.save(nib.Nifti1Image(pred, seg_img.affine, seg_img.header),
         OUT_3D / "BraTS20_Training_055_pred.nii")
nib.save(nib.Nifti1Image(error, seg_img.affine, seg_img.header),
         OUT_3D / "BraTS20_Training_055_error.nii")
print("saved to", OUT_3D)

saved to outputs/3d


All 12 test patients, saved the same way (overwrites any earlier volumes in `outputs/3d/`):

In [10]:
for case in TEST_CASES:
    if case in SCORES:
        continue
    pred, error, seg_img, score = predict_case(case)
    SCORES[case] = score
    nib.save(nib.Nifti1Image(pred, seg_img.affine, seg_img.header), OUT_3D / f"{case}_pred.nii")
    nib.save(nib.Nifti1Image(error, seg_img.affine, seg_img.header), OUT_3D / f"{case}_error.nii")
print("all volumes saved")

BraTS20_Training_068 slices: 48 mean Dice (cell-28 statistic): 0.88
BraTS20_Training_074 slices: 75 mean Dice (cell-28 statistic): 0.7083
BraTS20_Training_005 slices: 48 mean Dice (cell-28 statistic): 0.7494
BraTS20_Training_012 slices: 44 mean Dice (cell-28 statistic): 0.6775
BraTS20_Training_014 slices: 60 mean Dice (cell-28 statistic): 0.8784
BraTS20_Training_018 slices: 52 mean Dice (cell-28 statistic): 0.704
BraTS20_Training_029 slices: 63 mean Dice (cell-28 statistic): 0.8845
BraTS20_Training_032 slices: 43 mean Dice (cell-28 statistic): 0.8149
BraTS20_Training_036 slices: 33 mean Dice (cell-28 statistic): 0.8095
BraTS20_Training_004 slices: 64 mean Dice (cell-28 statistic): 0.8609
BraTS20_Training_015 slices: 76 mean Dice (cell-28 statistic): 0.8837
all volumes saved


Final check — every patient's score against code cell 28's recorded values:

In [11]:
EXPECTED = {
    "BraTS20_Training_068": 0.8800396141596138,
    "BraTS20_Training_074": 0.7083235201239586,
    "BraTS20_Training_005": 0.7494194824248552,
    "BraTS20_Training_055": 0.884770545980026,
    "BraTS20_Training_012": 0.677544023164294,
    "BraTS20_Training_014": 0.8784045338630676,
    "BraTS20_Training_018": 0.7040432771811118,
    "BraTS20_Training_029": 0.884545676291935,
    "BraTS20_Training_032": 0.8148784334229868,
    "BraTS20_Training_036": 0.8095457860917756,
    "BraTS20_Training_004": 0.8609172746073455,
    "BraTS20_Training_015": 0.8836845392851453,
}

for case in TEST_CASES:
    diff = abs(SCORES[case] - EXPECTED[case])
    print(case, "score:", round(SCORES[case], 4), "expected:", round(EXPECTED[case], 4), "diff:", f"{diff:.6f}")

BraTS20_Training_068 score: 0.88 expected: 0.88 diff: 0.000000
BraTS20_Training_074 score: 0.7083 expected: 0.7083 diff: 0.000000
BraTS20_Training_005 score: 0.7494 expected: 0.7494 diff: 0.000000
BraTS20_Training_055 score: 0.8848 expected: 0.8848 diff: 0.000000
BraTS20_Training_012 score: 0.6775 expected: 0.6775 diff: 0.000000
BraTS20_Training_014 score: 0.8784 expected: 0.8784 diff: 0.000000
BraTS20_Training_018 score: 0.704 expected: 0.704 diff: 0.000000
BraTS20_Training_029 score: 0.8845 expected: 0.8845 diff: 0.000000
BraTS20_Training_032 score: 0.8149 expected: 0.8149 diff: 0.000000
BraTS20_Training_036 score: 0.8095 expected: 0.8095 diff: 0.000000
BraTS20_Training_004 score: 0.8609 expected: 0.8609 diff: 0.000000
BraTS20_Training_015 score: 0.8837 expected: 0.8837 diff: 0.000000


In [13]:
def compare_case_128(case):
    seg, seg_img = load_seg(case)
    zs = evaluated_slices(seg)
    entries = [(case, z) for z in zs]
    ds = SliceDataset(entries, flip=False)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, multiprocessing_context="fork")

    comp = np.zeros((240, 240, 155), dtype=np.uint8)  # 0 neither, 1 both, 2 missed, 3 extra
    i = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            hard = (torch.sigmoid(logits) >= 0.5).float().cpu()
            for b in range(hard.size(0)):
                z = entries[i][1]
                i += 1
                p = hard[b, 0].numpy()
                t = y[b, 0].cpu().numpy()
                c = np.zeros((128, 128), dtype=np.uint8)
                c[(t == 1) & (p == 1)] = 1
                c[(t == 1) & (p == 0)] = 2
                c[(t == 0) & (p == 1)] = 3
                up = F.interpolate(torch.tensor(c).reshape(1, 1, 128, 128).float(),
                                   size=(240, 240), mode="nearest")
                comp[:, :, z] = up[0, 0].numpy().astype(np.uint8)

    nib.save(nib.Nifti1Image(comp, seg_img.affine, seg_img.header), OUT_3D / f"{case}_compare.nii")
    print(case, "both:", int((comp == 1).sum()), "missed:", int((comp == 2).sum()), "extra:", int((comp == 3).sum()))

for case in TEST_CASES:
    compare_case_128(case)


BraTS20_Training_068 both: 57944 missed: 5129 extra: 4359
BraTS20_Training_074 both: 55144 missed: 22105 extra: 3980
BraTS20_Training_005 both: 15779 missed: 6187 extra: 904
BraTS20_Training_055 both: 108613 missed: 9773 extra: 8402
BraTS20_Training_012 both: 25970 missed: 5839 extra: 6502
BraTS20_Training_014 both: 77613 missed: 2874 extra: 15416
BraTS20_Training_018 both: 38581 missed: 6330 extra: 11158
BraTS20_Training_029 both: 59188 missed: 3196 extra: 7766
BraTS20_Training_032 both: 24893 missed: 1877 extra: 5292
BraTS20_Training_036 both: 13971 missed: 1077 extra: 3853
BraTS20_Training_004 both: 94673 missed: 8610 extra: 9074
BraTS20_Training_015 both: 117921 missed: 7610 extra: 9615
